# Model Evaluation and Metrics — Exercises

Companion to the note [Model Evaluation and Metrics](Model%20Evaluation%20and%20Metrics.md).

Practise classification metrics from the confusion matrix (accuracy, precision, recall, F1, specificity, MCC), threshold-free metrics (ROC-AUC, PR-AUC / average precision, log loss), regression metrics, clustering metrics (silhouette, ARI) and ranking metrics (precision@k, MAP, MRR, NDCG): by hand and from scratch, checked against sklearn.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×7 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.metrics import (precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, roc_curve,
                             average_precision_score, log_loss, mean_squared_error, mean_absolute_error,
                             mean_absolute_percentage_error, r2_score, silhouette_score, adjusted_rand_score, ndcg_score,
                             confusion_matrix)
from sklearn.datasets import make_classification, make_blobs
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
# shared confusion matrix for Part B
TP, FP, FN, TN = 40, 10, 20, 930
y_cm = np.r_[np.ones(TP), np.zeros(FP), np.ones(FN), np.zeros(TN)]
p_cm = np.r_[np.ones(TP), np.ones(FP), np.zeros(FN), np.zeros(TN)]

## Part A · Concepts

### Exercise 1 · Which metric?
*🧠 Concept · ★☆☆*

Pick a primary metric from the note's table and justify it.

(a) Cancer screening, where follow-up tests are cheap. (b) A spam filter that must almost never delete a real e-mail.
(c) Fraud detection with 0.1 % positives where analysts review the top-scored transactions. (d) A weather model whose probabilities
are fed into a cost calculation. (e) Balanced 10-class digit recognition.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Which error is expensive (FP or FN)? Do you need a threshold, a ranking, or calibrated probabilities?

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Recall** (missing a cancer is costly, false alarms are cheap), reported together with precision.
(b) **Precision** of the spam class (a FP = real mail deleted). (c) **PR-AUC** or precision@k: heavy imbalance and a ranking task.
(d) **Log loss** (or Brier score): rewards calibrated probabilities. (e) **Accuracy** is fine for balanced classes, plus a confusion matrix.

</details>

### Exercise 2 · ROC-AUC as a probability
*🧠 Concept · ★★☆*

(a) Explain why ROC-AUC equals the probability that a randomly chosen positive gets a higher score than a randomly chosen negative.
(b) Why is ROC-AUC unchanged if you apply any strictly increasing function to the scores? (c) Why can a model have ROC-AUC 0.95 but
precision 0.1 at every useful threshold? What does PR-AUC show instead?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The ROC curve depends on the scores only through their ordering. FPR divides by the number of negatives.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Sweeping the threshold down, each negative passed moves the curve right by $1/N_-$; the height at that moment is the fraction of positives
ranked above it. Integrating gives $\frac{1}{N_+N_-}\sum_{i\in+}\sum_{j\in-}[s_i > s_j]$ (the Mann–Whitney $U$ statistic, normalised).
(b) It depends only on the ranking. (So it says nothing about calibration.)
(c) With 1 positive per 1000 negatives, an FPR of 1 % is already 10 false positives per true one. ROC hides this because FPR is normalised by the
huge negative class; precision is not, so the PR curve exposes it.

</details>

### Exercise 3 · Regression metrics: which and when
*🧠 Concept · ★☆☆*

Compare MSE, RMSE, MAE, $R^2$ and MAPE: units, sensitivity to outliers, and failure modes. Why is MAPE problematic for targets near zero,
and why is it asymmetric?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the denominator of MAPE, $|y_i|$, and at what happens when you over- vs under-predict a positive quantity.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MSE: squared units, heavily penalises large errors. RMSE: same ranking but in target units. MAE: target units, robust, optimal predictor is the median.
$R^2$: unitless, relative to predicting the mean; can be negative on test data. MAPE: scale-free percentage, but explodes (or is undefined) when $y_i\approx0$,
and under-predictions are bounded by 100 % while over-predictions are unbounded, so it rewards models that predict low.

</details>

### Exercise 4 · Internal vs. external clustering metrics
*🧠 Concept · ★★☆*

The note lists silhouette, Davies–Bouldin, adjusted Rand index and NMI. Which need ground-truth labels? Why is the Rand index *adjusted*
for chance, and why can silhouette prefer the "wrong" clustering on non-convex data such as two moons? See [[Clustering]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Silhouette compares within-cluster and nearest-other-cluster distances: what shape does that favour?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Silhouette and Davies–Bouldin are **internal** (geometry only); ARI and NMI are **external** (compare with true labels, invariant to label permutation).
The raw Rand index of two random labelings is far above 0 (most pairs are "different cluster" in both), so ARI subtracts its expectation under random
labelings: 0 = chance, 1 = perfect. Silhouette rewards compact, well-separated **convex** blobs, so k-means' straight cut through two moons can score
higher than the true moon-shaped clusters.

</details>

## Part B · By hand

### Exercise 5 · From a confusion matrix
*✍️ By hand · ★☆☆*

A classifier on 1000 cases has $TP=40$, $FP=10$, $FN=20$, $TN=930$. Compute accuracy, precision, recall, F1 and specificity.
Compare accuracy with the majority-class baseline.

In [ ]:
acc = None
prec = None
rec = None
f1 = None
spec = None

check('accuracy', acc, np.mean(y_cm == p_cm)); check('precision', prec, precision_score(y_cm, p_cm))
check('recall', rec, recall_score(y_cm, p_cm)); check('F1', f1, f1_score(y_cm, p_cm))
check('specificity', spec, recall_score(y_cm, p_cm, pos_label=0))

<details>
<summary><b>💡 Hint</b></summary>

F1 is the harmonic mean $\frac{2PR}{P+R}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Accuracy $= 970/1000 = 0.97$; precision $= 40/50 = 0.8$; recall $= 40/60 \approx 0.667$; F1 $= \frac{2\cdot0.8\cdot0.667}{1.467}\approx0.727$;
specificity $= 930/940\approx0.989$. The majority baseline ("always negative") already gets $940/1000 = 0.94$, so 0.97 is less impressive than it sounds.

```python
acc = 0.97
prec = 0.8
rec = 40 / 60
f1 = 2 * 0.8 * (2/3) / (0.8 + 2/3)
spec = 930 / 940
```

</details>

### Exercise 6 · Matthews correlation coefficient
*✍️ By hand · ★★☆*

For the same matrix compute
$\mathrm{MCC} = \dfrac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$.
What is the MCC of the "always negative" classifier, and why is that a useful property?

In [ ]:
mcc_hand = None

check('MCC', mcc_hand, matthews_corrcoef(y_cm, p_cm), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Numerator $40\cdot930 - 10\cdot20 = 37\,000$; the four sums are $50, 60, 940, 950$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{MCC} = 37000/\sqrt{50\cdot60\cdot940\cdot950} = 37000/51759 \approx 0.715$.
For "always negative" $TP = FP = 0$, so the numerator is 0 (sklearn returns 0 for the undefined $0/0$): MCC is a correlation between predictions and labels and
scores trivial classifiers at 0 regardless of imbalance, unlike accuracy (0.94).

```python
mcc_hand = 37000 / np.sqrt(50 * 60 * 940 * 950)
```

</details>

### Exercise 7 · ROC-AUC by counting pairs
*✍️ By hand · ★★☆*

Scores of 3 positives: $(0.9, 0.6, 0.4)$; scores of 4 negatives: $(0.7, 0.3, 0.2, 0.1)$.
Compute the ROC-AUC by counting correctly ordered (positive, negative) pairs.

In [ ]:
auc_hand = None

check('ROC-AUC', auc_hand, roc_auc_score([1,1,1,0,0,0,0], [.9,.6,.4,.7,.3,.2,.1]))

<details>
<summary><b>💡 Hint</b></summary>

There are $3\cdot4 = 12$ pairs. Which positive is ranked below which negative?

</details>

<details>
<summary><b>✅ Solution</b></summary>

0.9 beats all 4 negatives, 0.6 beats 3 (not 0.7), 0.4 beats 3 (not 0.7): $10/12\approx0.833$. Ties would count $\tfrac12$.

```python
auc_hand = 10 / 12
```

</details>

### Exercise 8 · Log loss of a base-rate predictor
*✍️ By hand · ★★☆*

10 % of the samples are positive. (a) What is the log loss of the constant prediction $\hat p = 0.1$? (b) Of $\hat p = 0.5$?
(c) One sample with $y=1$ gets $\hat p = 0.01$: what is its individual loss? What does this say about over-confident models?

In [ ]:
ll_base = None
ll_half = None
ll_wrong = None

_yb = np.r_[np.ones(10), np.zeros(90)]
check('constant 0.1', ll_base, log_loss(_yb, np.full(100, 0.1)))
check('constant 0.5', ll_half, log_loss(_yb, np.full(100, 0.5)))
check('confidently wrong', ll_wrong, log_loss([1], [0.01], labels=[0, 1]))

<details>
<summary><b>💡 Hint</b></summary>

(a) is the binary entropy $H(0.1) = -(0.1\ln0.1 + 0.9\ln0.9)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $H(0.1) \approx 0.325$: the baseline any useful probabilistic model must beat. (b) $\ln2\approx0.693$. (c) $-\ln0.01\approx4.61$, the cost of
14 base-rate predictions in a single sample. Log loss punishes confident mistakes without bound, so it rewards calibration.

```python
ll_base = -(0.1 * np.log(0.1) + 0.9 * np.log(0.9))
ll_half = np.log(2)
ll_wrong = -np.log(0.01)
```

</details>

### Exercise 9 · Regression metrics by hand
*✍️ By hand · ★☆☆*

$y = (100, 200, 300)$, $\hat y = (110, 190, 330)$. Compute MSE, RMSE, MAE, MAPE (as a fraction) and $R^2$.

In [ ]:
mse = None
rmse = None
mae = None
mape = None
r2 = None

_y, _yh = np.array([100, 200, 300.]), np.array([110, 190, 330.])
check('MSE', mse, mean_squared_error(_y, _yh)); check('RMSE', rmse, np.sqrt(mean_squared_error(_y, _yh)))
check('MAE', mae, mean_absolute_error(_y, _yh)); check('MAPE', mape, mean_absolute_percentage_error(_y, _yh))
check('R^2', r2, r2_score(_y, _yh))

<details>
<summary><b>💡 Hint</b></summary>

Errors $(10, -10, 30)$; $SS_{tot} = 100^2 + 0 + 100^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

MSE $= (100+100+900)/3 \approx 366.7$, RMSE $\approx 19.15$, MAE $= 50/3\approx16.67$, MAPE $= (0.1 + 0.05 + 0.1)/3 \approx 0.0833$,
$R^2 = 1 - 1100/20000 = 0.945$. RMSE > MAE because the single error of 30 dominates the squares.

```python
mse = 1100 / 3
rmse = np.sqrt(1100 / 3)
mae = 50 / 3
mape = 0.25 / 3
r2 = 1 - 1100 / 20000
```

</details>

### Exercise 10 · Precision depends on prevalence
*✍️ By hand · ★★☆*

A test has recall (TPR) $0.9$ and false-positive rate $0.05$. Compute its precision when the prevalence is (a) 50 % and (b) 1 %.
Which metrics in the note are unaffected by prevalence?

In [ ]:
prec_50 = None
prec_01 = None

def _prec(prev, tpr=0.9, fpr=0.05, n=10**6):
    pos = prev * n; neg = n - pos
    return tpr * pos / (tpr * pos + fpr * neg)
check('prevalence 50%', prec_50, _prec(0.5)); check('prevalence 1%', prec_01, _prec(0.01))

<details>
<summary><b>💡 Hint</b></summary>

Precision $= \dfrac{\text{TPR}\cdot\pi}{\text{TPR}\cdot\pi + \text{FPR}\cdot(1-\pi)}$ (Bayes' rule).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $0.45/(0.45 + 0.025)\approx0.947$. (b) $0.009/(0.009 + 0.0495)\approx0.154$: most alarms are false.
TPR (recall), FPR, specificity and hence ROC-AUC are conditioned on the true class and do **not** depend on prevalence; precision, accuracy, F1 and PR-AUC do.

```python
prec_50 = 0.45 / 0.475
prec_01 = 0.009 / 0.0585
```

</details>

### Exercise 11 · Ranking metrics by hand
*✍️ By hand · ★★☆*

A search engine returns 5 documents with relevance (in rank order) $(1, 0, 1, 0, 0)$; there are exactly 2 relevant documents.
Compute (a) precision@3, (b) average precision, (c) NDCG@5 with $\text{DCG} = \sum_i \frac{rel_i}{\log_2(i+1)}$.
(d) Over three queries the first relevant result is at ranks 1, 3 and 2: compute the MRR.

In [ ]:
p_at_3 = None
ap = None
ndcg = None
mrr = None

_rel = np.array([[1, 0, 1, 0, 0]]); _sc = np.array([[5, 4, 3, 2, 1]])
check('P@3', p_at_3, _rel[0, :3].mean()); check('AP', ap, average_precision_score(_rel[0], _sc[0]))
check('NDCG@5', ndcg, ndcg_score(_rel, _sc)); check('MRR', mrr, np.mean([1/1, 1/3, 1/2]))

<details>
<summary><b>💡 Hint 1</b></summary>

AP averages precision@k over the ranks $k$ of the relevant documents: ranks 1 and 3.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Ideal ranking: $(1,1,0,0,0)$, so IDCG $= 1 + 1/\log_2 3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $2/3$. (b) $\tfrac12(1/1 + 2/3) \approx 0.833$. (c) DCG $= 1 + 1/\log_2 4 = 1.5$, IDCG $= 1 + 1/\log_2 3 \approx 1.631$, NDCG $\approx 0.920$.
(d) $(1 + 1/3 + 1/2)/3 \approx 0.611$.

```python
p_at_3 = 2 / 3
ap = (1 + 2/3) / 2
ndcg = 1.5 / (1 + 1 / np.log2(3))
mrr = (1 + 1/3 + 1/2) / 3
```

</details>

## Part C · Code from scratch

### Exercise 12 · Metrics from the confusion matrix
*💻 Code · ★☆☆*

Implement `binary_metrics(y, yhat)` returning a dict with keys `acc, precision, recall, f1, specificity, mcc` (compute TP, FP, FN, TN yourself).

In [ ]:
y_m = rng.integers(0, 2, 300)
yhat_m = np.where(rng.random(300) < 0.8, y_m, 1 - y_m)

def binary_metrics(y, yhat):
    return None

bm = binary_metrics(y_m, yhat_m)

if bm is not None:
    check('precision', bm['precision'], precision_score(y_m, yhat_m)); check('recall', bm['recall'], recall_score(y_m, yhat_m))
    check('F1', bm['f1'], f1_score(y_m, yhat_m)); check('specificity', bm['specificity'], recall_score(y_m, yhat_m, pos_label=0))
    check('MCC', bm['mcc'], matthews_corrcoef(y_m, yhat_m)); check('accuracy', bm['acc'], np.mean(y_m == yhat_m))
else:
    check('metrics', None, 0)

<details>
<summary><b>💡 Hint</b></summary>

`TP = np.sum((y == 1) & (yhat == 1))` and similarly for the others.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Everything in the classification table derives from the four counts; only threshold-free metrics need the scores.

```python
y_m = rng.integers(0, 2, 300)
yhat_m = np.where(rng.random(300) < 0.8, y_m, 1 - y_m)

def binary_metrics(y, yhat):
    TP = np.sum((y == 1) & (yhat == 1)); TN = np.sum((y == 0) & (yhat == 0))
    FP = np.sum((y == 0) & (yhat == 1)); FN = np.sum((y == 1) & (yhat == 0))
    P, R = TP / (TP + FP), TP / (TP + FN)
    return dict(acc=(TP + TN) / len(y), precision=P, recall=R, f1=2 * P * R / (P + R),
                specificity=TN / (TN + FP),
                mcc=(TP * TN - FP * FN) / np.sqrt(float((TP + FP) * (TP + FN) * (TN + FP) * (TN + FN))))

bm = binary_metrics(y_m, yhat_m)
```

</details>

### Exercise 13 · ROC curve and AUC from scratch
*💻 Code · ★★☆*

Implement `roc_points(y, s)` returning `(fpr, tpr)` by sweeping the threshold over the sorted unique scores (start at $(0,0)$),
and `auc_trapezoid(fpr, tpr)`. Also implement `auc_rank(y, s)` via the pair-counting formula (ties count ½). All should agree with sklearn.

In [ ]:
X_c, y_c = make_classification(n_samples=600, weights=[0.8], random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_c, y_c, random_state=0)
s_c = np.round(LogisticRegression().fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1], 2)   # rounding creates ties

def roc_points(y, s):
    return None

def auc_trapezoid(fpr, tpr):
    return None

def auc_rank(y, s):
    return None

roc_c = roc_points(yc_te, s_c)

check('AUC (trapezoid)', None if roc_c is None else auc_trapezoid(*roc_c), roc_auc_score(yc_te, s_c))
check('AUC (pairs)', auc_rank(yc_te, s_c), roc_auc_score(yc_te, s_c))

<details>
<summary><b>💡 Hint 1</b></summary>

For each threshold $t$ in descending unique scores: TPR $= \frac{\#\{s\ge t, y=1\}}{N_+}$, FPR likewise for negatives.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Pairs: `(s[y==1][:, None] > s[y==0][None]).mean() + 0.5 * (… == …).mean()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Using unique thresholds makes tied scores move the curve diagonally, which is exactly the ½-credit for ties in the pair formula.

```python
X_c, y_c = make_classification(n_samples=600, weights=[0.8], random_state=1)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_c, y_c, random_state=0)
s_c = np.round(LogisticRegression().fit(Xc_tr, yc_tr).predict_proba(Xc_te)[:, 1], 2)

def roc_points(y, s):
    th = np.unique(s)[::-1]
    P, N = np.sum(y == 1), np.sum(y == 0)
    tpr = np.r_[0, [np.sum((s >= t) & (y == 1)) / P for t in th]]
    fpr = np.r_[0, [np.sum((s >= t) & (y == 0)) / N for t in th]]
    return fpr, tpr

def auc_trapezoid(fpr, tpr):
    return np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2)

def auc_rank(y, s):
    sp, sn = s[y == 1][:, None], s[y == 0][None]
    return np.mean(sp > sn) + 0.5 * np.mean(sp == sn)

roc_c = roc_points(yc_te, s_c)
```

</details>

### Exercise 14 · Average precision (PR-AUC) from scratch
*💻 Code · ★★☆*

Implement `average_precision(y, s)` $= \sum_n (R_n - R_{n-1})\,P_n$ over thresholds at the sorted unique scores (as sklearn does).
Compare ROC-AUC and AP on the imbalanced test set from the previous exercise (≈ 20 % positives).

In [ ]:
def average_precision(y, s):
    return None

ap_c = average_precision(yc_te, s_c)

check('average precision', ap_c, average_precision_score(yc_te, s_c))

<details>
<summary><b>💡 Hint</b></summary>

At each threshold $t$ (descending): $P = TP/(TP+FP)$, $R = TP/N_+$; start from $R_0 = 0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Here ROC-AUC ≈ 0.91 but AP ≈ 0.74. AP is a step-wise (not interpolated) area under the PR curve. Under imbalance AP is typically well below ROC-AUC because it is sensitive to false positives among the
top-ranked items; with balanced classes the two are much closer. The random-classifier baseline of AP is the prevalence, not 0.5.

```python
def average_precision(y, s):
    th = np.unique(s)[::-1]
    P_tot = np.sum(y == 1)
    ap, r_prev = 0.0, 0.0
    for t in th:
        pred = s >= t
        tp = np.sum(pred & (y == 1))
        prec, rec = tp / pred.sum(), tp / P_tot
        ap += (rec - r_prev) * prec
        r_prev = rec
    return ap

ap_c = average_precision(yc_te, s_c)
print('imbalanced: ROC-AUC', round(roc_auc_score(yc_te, s_c), 3), 'AP', round(ap_c, 3), '| prevalence', round(yc_te.mean(), 3))
```

</details>

### Exercise 15 · Silhouette score from scratch
*💻 Code · ★★☆*

Implement `silhouette(X, labels)`: for each point $a(i)$ = mean distance to the other members of its cluster, $b(i)$ = smallest mean distance to
another cluster, $s(i) = \frac{b-a}{\max(a,b)}$ (0 for singleton clusters); return the mean. Use it to pick $k$ for k-means on 4 blobs.

In [ ]:
from sklearn.cluster import KMeans
X_bl, _ = make_blobs(300, centers=4, cluster_std=0.8, random_state=3)

def silhouette(X, labels):
    return None

lab3 = KMeans(3, n_init=10, random_state=0).fit_predict(X_bl)
sil3 = silhouette(X_bl, lab3)
best_k = None   # k in 2..6 with the highest silhouette

check('silhouette', sil3, silhouette_score(X_bl, lab3))
check('best k', best_k, 2 + int(np.argmax([silhouette_score(X_bl, KMeans(k, n_init=10, random_state=0).fit_predict(X_bl)) for k in range(2, 7)])))

<details>
<summary><b>💡 Hint</b></summary>

Full distance matrix `D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))`; for own-cluster mean divide by $n_c - 1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The silhouette peaks at $k=4$ (≈ 0.78), matching the generating blobs. (With other seeds, where two blobs overlap, it can prefer fewer clusters: it measures separation, not "truth".) It needs no labels (internal metric), but costs $O(N^2)$ distances.

```python
from sklearn.cluster import KMeans
X_bl, _ = make_blobs(300, centers=4, cluster_std=0.8, random_state=3)

def silhouette(X, labels):
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    ks = np.unique(labels)
    s = np.zeros(len(X))
    for i in range(len(X)):
        own = labels == labels[i]
        if own.sum() == 1:
            continue
        a = D[i, own].sum() / (own.sum() - 1)
        b = min(D[i, labels == k].mean() for k in ks if k != labels[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()

lab3 = KMeans(3, n_init=10, random_state=0).fit_predict(X_bl)
sil3 = silhouette(X_bl, lab3)
sils = {k: silhouette(X_bl, KMeans(k, n_init=10, random_state=0).fit_predict(X_bl)) for k in range(2, 7)}
best_k = max(sils, key=sils.get)
print({k: round(v, 3) for k, v in sils.items()})
```

</details>

### Exercise 16 · Adjusted Rand index from scratch
*💻 Code · ★★★*

Implement `ari(a, b)` from the contingency table $n_{ij}$ with row sums $a_i$ and column sums $b_j$:
$$\mathrm{ARI} = \frac{\sum_{ij}\binom{n_{ij}}{2} - \big[\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}\big]/\binom{n}{2}}
{\tfrac12\big[\sum_i\binom{a_i}{2} + \sum_j\binom{b_j}{2}\big] - \big[\sum_i\binom{a_i}{2}\sum_j\binom{b_j}{2}\big]/\binom{n}{2}}.$$
Check that a label permutation gives 1 and random labels give ≈ 0.

In [ ]:
truth = np.repeat([0, 1, 2], 50)
noisy = np.where(rng.random(150) < 0.8, truth, rng.integers(0, 3, 150))
permuted = np.array([2, 0, 1])[truth]
random_lab = rng.integers(0, 3, 150)

def ari(a, b):
    return None

check('ARI noisy', ari(truth, noisy), adjusted_rand_score(truth, noisy))
check('ARI permuted = 1', ari(truth, permuted), 1.0)
check('ARI random', ari(truth, random_lab), adjusted_rand_score(truth, random_lab))

<details>
<summary><b>💡 Hint</b></summary>

`C = confusion_matrix(a, b)` (or `np.add.at`); `comb2 = lambda n: n * (n - 1) / 2` works elementwise on arrays.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The index counts pairs of points on which the two labelings agree, then subtracts the expected count under random labelings with the same cluster sizes.
It is invariant to renaming clusters, which is why it suits clustering, where label IDs are arbitrary.

```python
truth = np.repeat([0, 1, 2], 50)
noisy = np.where(rng.random(150) < 0.8, truth, rng.integers(0, 3, 150))
permuted = np.array([2, 0, 1])[truth]
random_lab = rng.integers(0, 3, 150)

def ari(a, b):
    _, ai = np.unique(a, return_inverse=True); _, bi = np.unique(b, return_inverse=True)
    C = np.zeros((ai.max() + 1, bi.max() + 1)); np.add.at(C, (ai, bi), 1)
    c2 = lambda n: n * (n - 1) / 2
    sij, sa, sb = c2(C).sum(), c2(C.sum(1)).sum(), c2(C.sum(0)).sum()
    expected = sa * sb / c2(len(a))
    return (sij - expected) / (0.5 * (sa + sb) - expected)

print(ari(truth, noisy), ari(truth, random_lab))
```

</details>

### Exercise 17 · NDCG@k from scratch
*💻 Code · ★★☆*

Implement `ndcg_at_k(rel, scores, k)` for a **batch** of queries (2-D arrays): sort each row by score, compute DCG@k with gain $rel_i$ and discount
$1/\log_2(i+1)$, divide by the ideal DCG@k, and average over queries.

In [ ]:
rel_q = rng.integers(0, 4, size=(20, 10))            # graded relevance 0..3
scores_q = rel_q + rng.normal(scale=1.5, size=(20, 10))

def ndcg_at_k(rel, scores, k):
    return None

check('NDCG@5', ndcg_at_k(rel_q, scores_q, 5), ndcg_score(rel_q, scores_q, k=5))
check('NDCG@10', ndcg_at_k(rel_q, scores_q, 10), ndcg_score(rel_q, scores_q, k=10))

<details>
<summary><b>💡 Hint</b></summary>

`order = np.argsort(-scores, axis=1)`; `np.take_along_axis(rel, order, 1)[:, :k]`; discounts `1 / np.log2(np.arange(2, k + 2))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

NDCG rewards putting highly relevant items early, with logarithmically decaying credit, and normalises so a perfect ranking scores 1. It handles graded relevance,
unlike precision@k and MAP. (With continuous random scores there are no ties, so the result matches sklearn exactly.)

```python
rel_q = rng.integers(0, 4, size=(20, 10))
scores_q = rel_q + rng.normal(scale=1.5, size=(20, 10))

def ndcg_at_k(rel, scores, k):
    disc = 1 / np.log2(np.arange(2, k + 2))
    got = np.take_along_axis(rel, np.argsort(-scores, axis=1), 1)[:, :k]
    ideal = -np.sort(-rel, axis=1)[:, :k]
    dcg, idcg = got @ disc, ideal @ disc
    return np.mean(np.where(idcg > 0, dcg / np.where(idcg > 0, idcg, 1), 0))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Model Evaluation and Metrics](Model%20Evaluation%20and%20Metrics.md).*